In [2]:
# Install necessary packages

In [3]:
!pip3 install huggingface_hub
import pandas as pd 


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [4]:
# Load and inspect Dataset

In [5]:
p = pd.read_csv("hf://datasets/TheFusion21/PokemonCards/train.csv")

print("#####COLUMNS IN DATASET#####")
print(p.columns)

print("\n")

print("#####FIRST 5 ROWS OF DATASET#####")
print(p.head())

print("\n")

print("#####INFO OF DATASET#####")
print(p.info())

print("\n")

print("#####SHAPE OF DATASET#####")
print(p.shape)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#####COLUMNS IN DATASET#####
Index(['id', 'image_url', 'caption', 'name', 'hp', 'set_name'], dtype='object')


#####FIRST 5 ROWS OF DATASET#####
        id                                       image_url  \
0    pl3-1    https://images.pokemontcg.io/pl3/1_hires.png   
1   ex12-1   https://images.pokemontcg.io/ex12/1_hires.png   
2    xy5-1    https://images.pokemontcg.io/xy5/1_hires.png   
3  mcd19-1  https://images.pokemontcg.io/mcd19/1_hires.png   
4    ex7-1    https://images.pokemontcg.io/ex7/1_hires.png   

                                             caption        name  hp  \
0  A Basic, SP Pokemon Card of type Darkness with...     Absol G  70   
1  A Stage 1 Pokemon Card of type Colorless with ...  Aerodactyl  70   
2  A Basic Pokemon Card of type Grass with the ti...      Weedle  50   
3  A Basic Pokemon Card of type Grass with the ti...    Caterpie  50   
4  A Stage 1 Pokemon Card of type Water with the ...   Azumarill  80   

                     set_name  
0             Sup

In [6]:
# Check for Null Values

In [7]:
print("#####NULL VALUES IN DATASET PER COLUMN#####")
p.isnull().sum()

#####NULL VALUES IN DATASET PER COLUMN#####


id           0
image_url    0
caption      0
name         0
hp           0
set_name     0
dtype: int64

In [8]:
# Check for Duplicates

In [9]:
print("#####NUMBER OF DUPLICATE ROWS IN DATASET#####")
print(p.duplicated().sum())

print("\n")

print("#####NUMBER OF DUPLICATE ROWS BASED ON 'name', 'set_name', 'caption', 'hp' COLUMNS#####")
print(p.duplicated(subset=['name', 'set_name', 'caption', 'hp']).sum())

print("\n")

print("#####SAMPLE DUPLICATE ROWS BASED ON 'name', 'set_name', 'caption' AND 'hp' COLUMNS#####")
dupes = p[p.duplicated(subset=['name', 'set_name', 'caption', 'hp'], keep=False)]
dupes.sort_values(by=['name', 'set_name', 'caption', 'hp']).head(10)

#####NUMBER OF DUPLICATE ROWS IN DATASET#####
0


#####NUMBER OF DUPLICATE ROWS BASED ON 'name', 'set_name', 'caption', 'hp' COLUMNS#####
189


#####SAMPLE DUPLICATE ROWS BASED ON 'name', 'set_name', 'caption' AND 'hp' COLUMNS#####


,id,image_url,caption,name,hp,set_name
12840,swsh11-179,https://images.pokemontcg.io/swsh11/179_hires.png,"A Basic, V Pokemon Card of type Fighting with ...",Aerodactyl V,210,Lost Origin
12841,swsh11-180,https://images.pokemontcg.io/swsh11/180_hires.png,"A Basic, V Pokemon Card of type Fighting with ...",Aerodactyl V,210,Lost Origin
466,sm6-2,https://images.pokemontcg.io/sm6/2_hires.png,A Stage 1 Pokemon Card of type Grass with the ...,Alolan Exeggutor,160,Forbidden Light
676,sm6-2a,https://images.pokemontcg.io/sm6/2a_hires.png,A Stage 1 Pokemon Card of type Grass with the ...,Alolan Exeggutor,160,Forbidden Light
1832,sm2-19,https://images.pokemontcg.io/sm2/19_hires.png,A Basic Pokemon Card of type Water with the ti...,Alolan Sandshrew,60,Guardians Rising
1912,sm2-19a,https://images.pokemontcg.io/sm2/19a_hires.png,A Basic Pokemon Card of type Water with the ti...,Alolan Sandshrew,60,Guardians Rising
2053,sm2-21,https://images.pokemontcg.io/sm2/21_hires.png,A Basic Pokemon Card of type Water with the ti...,Alolan Vulpix,60,Guardians Rising
2208,sm2-21a,https://images.pokemontcg.io/sm2/21a_hires.png,A Basic Pokemon Card of type Water with the ti...,Alolan Vulpix,60,Guardians Rising
4077,sm75-40,https://images.pokemontcg.io/sm75/40_hires.png,A Stage 1 Pokemon Card of type Dragon with the...,Altaria,80,Dragon Majesty
4204,sm75-40a,https://images.pokemontcg.io/sm75/40a_hires.png,A Stage 1 Pokemon Card of type Dragon with the...,Altaria,80,Dragon Majesty


In [10]:
# Create New DataFrame

In [27]:
llava_df = pd.DataFrame()

llava_df["id"] = (p.index+1).astype(str).str.zfill(5)
llava_df['image'] = p["image_url"]
llava_df['input'] = "Can you write a detailed description of the following Pokemon card?"
llava_df['output'] = p["caption"]
llava_df["type"] = "conv"

llava_df_without_dupes = llava_df.drop_duplicates(subset=['output'])

print("#####SHAPE OF NEW DATAFRAME WITHOUT DUPLICATES#####")
print(llava_df_without_dupes.shape)

print("\n")

print("#####NEW DATAFRAME COLUMNS#####")
print(llava_df.columns)

print("\n")

print("#####FIRST ROW OF NEW DATAFRAME#####")
print(llava_df.iloc[0])

#####SHAPE OF NEW DATAFRAME WITHOUT DUPLICATES#####
(12950, 5)


#####NEW DATAFRAME COLUMNS#####
Index(['id', 'image', 'input', 'output', 'type'], dtype='object')


#####FIRST ROW OF NEW DATAFRAME#####
id                                                    00001
image          https://images.pokemontcg.io/pl3/1_hires.png
input     Can you write a detailed description of the fo...
output    A Basic, SP Pokemon Card of type Darkness with...
type                                                   conv
Name: 0, dtype: object


In [28]:
# Adjust URLs for direct image access
def adjust_image_url(url):
    return "_".join(url.split("/")[-2:]).split(".")[0] + ".jpg"

llava_df['image'] = llava_df['image'].apply(adjust_image_url)   
llava_df_without_dupes['image'] = llava_df_without_dupes['image'].apply(adjust_image_url)
print("#####ADJUSTED IMAGE URL SAMPLE#####")
print(llava_df['image'].head())

#####ADJUSTED IMAGE URL SAMPLE#####
0      pl3_1_hires.jpg
1     ex12_1_hires.jpg
2      xy5_1_hires.jpg
3    mcd19_1_hires.jpg
4      ex7_1_hires.jpg
Name: image, dtype: object


/var/folders/8k/s6fjz5gn2qgfs589f5g1l6940000gn/T/ipykernel_60613/1115999349.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  llava_df_without_dupes['image'] = llava_df_without_dupes['image'].apply(adjust_image_url)


In [ ]:
llava_conv = []

for _, row in llava_df.iterrows():
    llava_conv.append({
        "id": row['id'],
        "image": row['image'],
        "conversations": [
            {
                "from": "human", 
                "value": "<image>\n" + row['input']
            },
            {
                "from": "gpt", 
                "value": row['output']
            }
        ]
    })

print("#####SAMPLE CONVERSATION FORMAT#####")
print(llava_conv[0])

llava_conv_without_dupes = []

for _, row in llava_df_without_dupes.iterrows():
    llava_conv_without_dupes.append({
        "id": row['id'],
        "image": row['image'],
        "conversations": [
            {
                "from": "human", 
                "value": "<image>\n" + row['input']
            },
            {
                "from": "gpt", 
                "value": row['output']
            }
        ]   
    })

print("#####SAMPLE CONVERSATION FORMAT WITHOUT DUPLICATES#####")
print(llava_conv_without_dupes[0])

#####SAMPLE CONVERSATION FORMAT#####
{'id': '00001', 'image': 'pl3_1_hires.jpg', 'conversations': [{'from': 'human', 'value': '<image>\nCan you write a detailed description of the following Pokemon card?'}, {'from': 'gpt', 'value': "A Basic, SP Pokemon Card of type Darkness with the title Absol G and 70 HP of rarity Rare Holo from the set Supreme Victors.  It has the attack Feint Attack with the cost Darkness, the energy cost 1 with the description: Choose 1 of your opponent's Pokemon. This attack does 20 damage to that Pokemon. This attack's damage isn't affected by Weakness, Resistance, Poke-Powers, Poke-Bodies, or any other effects on that Pokemon. It has the attack Doom News with the cost Darkness, Colorless, Colorless, the energy cost 3 with the description: Return all Energy cards attached to Absol G to your hand. The Defending Pokemon is Knocked Out at the end of your opponent's next turn. It has weakness against Fighting 2. It has resistance against Psychic -20. "}]}
#####SAMPL

In [31]:
# split into train and test sets
llava_df_TRAIN = llava_conv[:10000]
llava_df_TEST = llava_conv[10000:]

llava_df_without_dupes_TRAIN = llava_conv_without_dupes[:10000]
llava_df_without_dupes_TEST = llava_conv_without_dupes[10000:]

print("#####SHAPE OF TRAIN AND TEST SETS#####")
print("TRAIN SET SHAPE WITH DUPES: ", len(llava_df_TRAIN), len(llava_df_TRAIN[0]))
print("TEST SET SHAPE WITH DUPES: ", len(llava_df_TEST), len(llava_df_TEST[0]))
print("TRAIN SET SHAPE WITHOUT DUPES: ", len(llava_df_without_dupes_TRAIN), len(llava_df_without_dupes_TRAIN[0]))
print("TEST SET SHAPE WITHOUT DUPES: ", len(llava_df_without_dupes_TEST), len(llava_df_without_dupes_TEST[0]))

#####SHAPE OF TRAIN AND TEST SETS#####
TRAIN SET SHAPE WITH DUPES:  10000 3
TEST SET SHAPE WITH DUPES:  3139 3
TRAIN SET SHAPE WITHOUT DUPES:  10000 3
TEST SET SHAPE WITHOUT DUPES:  2950 3


In [32]:
# Save New DataFrame to jsonl
import json

with open("pokemon_llava_dataset_train.jsonl", "w") as f:
    for entry in llava_df_TRAIN:
        f.write(json.dumps(entry) + "\n")

with open("pokemon_llava_dataset_without_dupes_train.jsonl", "w") as f:
    for entry in llava_df_without_dupes_TRAIN:
        f.write(json.dumps(entry) + "\n")    

with open("pokemon_llava_dataset_test.jsonl", "w") as f:
    for entry in llava_df_TEST:
        f.write(json.dumps(entry) + "\n")

with open("pokemon_llava_dataset_without_dupes_test.jsonl", "w") as f:
    for entry in llava_df_without_dupes_TEST:
        f.write(json.dumps(entry) + "\n")